# 코트 키포인트 모델 학습

1. **런타임 > 런타임 유형 변경 > GPU**로 바꿉니다.
2. Google Drive의 `MyDrive/court_pose/`에 `court_pose_dataset.zip`을 올립니다.
3. 위에서부터 차례로 실행합니다. 학습 결과는 Drive에 저장되므로 런타임이 끊겨도 **이어서 학습** 셀로 계속할 수 있습니다.

데이터에는 경기 영상 속 사람이 찍혀 있으니 Drive 폴더는 공개로 공유하지 마세요.


In [ ]:
# GPU 확인
!nvidia-smi -L


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/court_pose'


In [ ]:
# 프로젝트와 같은 버전의 ultralytics 설치, 데이터 압축 해제
!pip install -q ultralytics==8.4.183
!rm -rf /content/court_pose && mkdir -p /content/court_pose
!unzip -q -o {DRIVE}/court_pose_dataset.zip -d /content/court_pose
!ls /content/court_pose/images/train | wc -l
!ls /content/court_pose/images/val | wc -l
!cat /content/court_pose/data.yaml


## 학습
화면 전체의 코트 기하가 중요하므로 mosaic과 회전 증강은 끕니다.

**좌우 반전은 켭니다 (fliplr=0.5).** 1103 촬영은 카메라가 반대쪽 모서리에 서서 화면이 1008의 좌우 반전이고, 두 배치를 모두 다뤄야 합니다. 지점 이름은 화면 기준 왼쪽/오른쪽이라 반전하면 이름도 바뀌어야 하며, data.yaml의 flip_idx가 그 짝을 알려 줍니다 (make_dataset.FLIP_IDX, 테스트로 확인). 1008만으로 학습한 첫 모델은 fliplr=0.5에서 먼 쪽 오른쪽 지점이 560~690 px 틀렸으므로, 학습 뒤 eval로 두 회차 모두 확인합니다.

1103은 카메라가 더 높고 멀어서 크기·원근 증강을 키웠습니다 (scale 0.3, perspective 0.0003).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')
model.train(
    data='/content/court_pose/data.yaml',
    imgsz=960, epochs=100, batch=16, patience=25,
    mosaic=0.0, degrees=0.0, shear=0.0, perspective=0.0003,
    translate=0.1, scale=0.3, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    project=DRIVE + '/runs', name='court_pose', exist_ok=True, save_period=10,
)


## 이어서 학습 (런타임이 끊겼을 때만)
위의 드라이브 연결과 설치 셀을 다시 실행한 뒤 이 셀을 실행합니다.


In [ ]:
# from ultralytics import YOLO
# YOLO(DRIVE + '/runs/court_pose/weights/last.pt').train(resume=True)


## 검증
결과 파일 `best.pt`를 `MyDrive/court_pose/runs/court_pose/weights/`에서 받아 프로젝트의 `train/best.pt`에 넣습니다.


In [ ]:
from ultralytics import YOLO
best = YOLO(DRIVE + '/runs/court_pose/weights/best.pt')
print(best.val(data='/content/court_pose/data.yaml', imgsz=960).pose.map50)
